# Credit ratios from a statement model

Compute **leverage**, **coverage**, and **liquidity** from evaluated nodes, then layer `run_corporate_analysis`, `pl_summary_report`, and `credit_assessment_report` when available.

## Concept

Credit workpapers typically pair **P&L** (EBITDA, interest) with **balance sheet** (debt, cash). Scenario overlays (`evaluate_scenario_set`) stress those drivers. Ratios below are **illustrative** and should be aligned to your facility definitions (cash taxes, lease adjustments, etc.).

In [ ]:
from finstack.statements import ModelBuilder, Evaluator, FinancialModelSpec
from finstack.statements_analytics import run_sensitivity, evaluate_scenario_set, goal_seek, trace_dependencies, explain_formula, run_variance, evaluate_dcf, run_corporate_analysis, pl_summary_report, credit_assessment_report
import json
print("Loaded finstack.statements + finstack.statements_analytics")

from finstack.statements_analytics import (
    run_corporate_analysis,
    pl_summary_report,
    credit_assessment_report,
    evaluate_scenario_set,
)

PERIODS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]


def z(v):
    return list(zip(PERIODS, v))


b = ModelBuilder("credit-demo")
b.periods("2025Q1..Q4", "2025Q2")
b.value("revenue", z([200.0, 210.0, 218.0, 225.0]))
b.compute("ebitda", "revenue * 0.18")
b.value("interest_expense", z([7.0, 7.2, 7.5, 7.8]))
b.compute("ebit", "ebitda - interest_expense")
b.value("principal_paid", z([5.0, 5.0, 5.5, 5.5]))
b.value("total_debt", z([120.0, 118.0, 115.0, 112.0]))
b.value("cash", z([15.0, 16.0, 17.0, 18.0]))
b.compute("net_debt", "total_debt - cash")
b.value("capex", z([8.0, 8.5, 9.0, 9.0]))
b.compute("free_cash_flow", "ebitda - interest_expense - principal_paid - capex")

spec = b.build()
res = Evaluator().evaluate(spec)
eval_json = res.to_json()
model_json = spec.to_json()

as_of = "2025Q4"
td = res.get("total_debt", as_of) or 0.0
eb = res.get("ebitda", as_of) or 0.0
ie = res.get("interest_expense", as_of) or 0.0
prin = res.get("principal_paid", as_of) or 0.0
capex_val = res.get("capex", as_of) or 0.0

leverage = td / eb if eb > 0 else float("nan")
interest_cov = eb / ie if ie else float("nan")
fCCR = (eb - capex_val) / (ie + prin) if (ie + prin) else float("nan")
cash_m = res.get("cash", as_of) or 0.0
liquidity = cash_m / ie if ie else float("nan")

print("Debt / EBITDA:", round(leverage, 2), "x")
print("EBITDA / Interest:", round(interest_cov, 2), "x")
print("FCCR ((EBITDA - Capex) / (Int+Prin)):", round(fCCR, 2), "x")
print("Cash / Interest (liquidity proxy):", round(liquidity, 2), "x")

print(pl_summary_report(eval_json, ["revenue", "ebitda", "interest_expense"], PERIODS))
print(credit_assessment_report(eval_json, as_of))


In [ ]:
print("Corporate analysis + scenarios")
tv = json.dumps({"type": "gordon_growth", "growth_rate": 0.02})
raw = json.loads(model_json)
raw.setdefault("meta", {})["currency"] = "USD"
mj = FinancialModelSpec.from_json(json.dumps(raw)).to_json()

try:
    corp = run_corporate_analysis(
        mj,
        wacc=0.10,
        terminal_value_json=tv,
        net_debt_override=95.0,
        coverage_node="ebitda",
    )
    print("run_corporate_analysis keys:", sorted(corp.keys()))
    if "equity" in corp:
        print("equity:", corp["equity"])
except Exception as e:
    print("run_corporate_analysis skipped:", e)

scenarios = json.dumps(
    {
        "scenarios": {
            "base": {"overrides": {}},
            "recession": {"overrides": {"revenue": 180.0}},
        }
    }
)
out = evaluate_scenario_set(model_json, scenarios)
print("evaluate_scenario_set output (prefix):", out[:400], "...")
parsed = json.loads(out)
print("Scenario names:", list(parsed.keys()))


## Takeaways

- **Leverage and coverage** are node-for-node: align names like `total_debt`, `ebitda`, and `interest_expense` with your model.
- `credit_assessment_report` is a **fast text view**; pair with custom ratios for **FCCR / DSCR** per facility.
- `evaluate_scenario_set` applies **forecast overrides** so you can stress revenue and re-read ratios.